# Gale demo: one-sol dust devil likelihood map (per pixel)

**What this is:** a *heuristic* 0-1 score per pixel for one sol, built from terrain, albedo, thermal inertia and modeled sunlight. It is **not** a calibrated probability and **not** detected dust devils.

**Why not detections:** `rems_binned_all_sols.csv` holds one mean pressure per 30-minute bin (plus a sample count). A vortex lasts seconds to minutes, so a bin mean hides it. Real detection needs raw 1 Hz REMS pressure (RMD files). Until then the layer stays a likelihood.

Run top to bottom. Terrain, thermal inertia and albedo now come from one CSV (`gale_dust_devil_features_100m.csv`, built by `../data_prep/build_dust_devil_feature_table.ipynb`), not from the separate rasters.

Needs: numpy and pandas only. **No rasterio, matplotlib or BLAS-heavy calls here on purpose**: this kernel crashed (exit code 3228369023 = 0xC06D007F, two clashing native libraries) once rasterio was loaded next to numpy's heavy maths.

Run order: this notebook -> `write_likelihood_geotiff.ipynb` (rasterio, own kernel) and `dust_devil_overlay_preview.ipynb` (matplotlib, own kernel).

In [8]:
from pathlib import Path
import json
import numpy as np, pandas as pd

PROC = Path("../processed_data")          # folder with your .tif/.csv files
BINNED_CSV = PROC / "rems_binned_all_sols.csv"
FEATURES_CSV = PROC / "gale_dust_devil_features_100m.csv"      # one row per 100 m pixel
META_JSON = PROC / "gale_dust_devil_meta.json"                 # template grid (shape + transform)

DEMO_SOL = 800        # pick a sol with many bins; the next cell prints which sols have >=40 bins
TAU = 0.5             # PLACEHOLDER dust opacity; replace with the Mastcam tau for the sol (see TAU_CSV below)
TAU_CSV = None        # optional CSV with columns sol,tau
SLOPE_SCALE_DEG = 10.0   # score falls off for slopes steeper than this
DAY_LTST = (9.0, 16.0)   # window used for the daily-mean score
NPZ = PROC / f"gale_dust_devil_likelihood_sol{DEMO_SOL}.npz"   # read by the GeoTIFF and overlay notebooks

In [9]:
d = pd.read_csv(BINNED_CSV)
per_sol = d.groupby("sol").agg(bins=("bin","size"), lat=("latitude_deg","first"), lon=("longitude_deg_e","first"),
                               elev=("elevation_m","first"))
print(f"{len(per_sol)} sols in file, sol {per_sol.index.min()}..{per_sol.index.max()}")
print("sols with >=40 bins (bins are 15 min, max 96/day):", per_sol.index[per_sol.bins >= 40].tolist())
day = d[d.sol == DEMO_SOL].sort_values("bin").reset_index(drop=True)
assert len(day) > 0, f"sol {DEMO_SOL} not in file"
print(f"\nDemo sol {DEMO_SOL}: {len(day)} bins, rover at lat {day.latitude_deg[0]:.5f}, lon {day.longitude_deg_e[0]:.5f}, elev {day.elevation_m[0]:.1f} m")
day["gap_obs_k"] = day.ground_temp_k - day.air_temp_k
day[["ltst_bin_center_hr","air_temp_k","ground_temp_k","gap_obs_k","pressure_pa"]].head(8)

115 sols in file, sol 1..3778
sols with >=40 bins (bins are 15 min, max 96/day): [21, 22, 23, 26, 30, 33, 34, 35, 36, 37, 38, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 368, 584, 585, 590, 592, 709, 710, 799, 800, 801, 802, 803, 804, 1161, 1162, 1163, 1343, 1347, 1348, 1349, 1350, 1352, 1355, 1356, 1357, 1358, 1359, 1360, 3645, 3646, 3647, 3648, 3649, 3650, 3651, 3652, 3653, 3655, 3763, 3764, 3765, 3766, 3767, 3768, 3770, 3771, 3773, 3776, 3777, 3778]

Demo sol 800: 60 bins, rover at lat -4.66257, lon 137.37752, elev -4461.2 m


,ltst_bin_center_hr,air_temp_k,ground_temp_k,gap_obs_k,pressure_pa
0,0.38,216.72,209.20,-7.52,914.57
1,0.62,216.72,209.93,-6.79,915.08
2,1.38,216.43,207.11,-9.32,915.60
3,1.62,216.28,208.20,-8.08,915.41
4,2.38,214.42,205.55,-8.87,913.27
5,2.62,213.69,205.51,-8.18,913.41
6,3.38,210.31,204.64,-5.67,910.79
7,3.62,209.51,203.29,-6.22,910.46


In [10]:
# ---- Solar geometry (Allison & McEwen 2000 style; good to ~0.1 deg) ----
SOL_DAYS = 1.0274912517
LANDING_JD_TT = 2456145.7 + 69.184/86400     # MSL landing 2012-08-06, sol 0 (approx.)
def ls_deg(sol):
    jd = LANDING_JD_TT + np.asarray(sol, float) * SOL_DAYS
    dj = jd - 2451545.0
    M = np.radians(19.3871 + 0.52402073 * dj)
    alpha = 270.3863 + 0.52403840 * dj
    pbs = (0.0071*np.cos(np.radians(0.985626*dj/2.2353+49.409)) + 0.0057*np.cos(np.radians(0.985626*dj/2.7543+168.173))
           + 0.0039*np.cos(np.radians(0.985626*dj/1.1177+191.837)) + 0.0037*np.cos(np.radians(0.985626*dj/15.7866+21.736))
           + 0.0021*np.cos(np.radians(0.985626*dj/2.1354+15.704)) + 0.0020*np.cos(np.radians(0.985626*dj/2.4694+95.528))
           + 0.0018*np.cos(np.radians(0.985626*dj/32.8493+49.095)))
    eoc = (10.691 + 3.0e-7*dj)*np.sin(M) + 0.623*np.sin(2*M) + 0.050*np.sin(3*M) + 0.005*np.sin(4*M) + 0.0005*np.sin(5*M) + pbs
    return (alpha + eoc) % 360

def sun_vector(ls, lat_deg, ltst_hr):
    """Unit vector to the Sun in local (East, North, Up). lat_deg may be an array."""
    dec = np.arcsin(np.sin(np.radians(25.19)) * np.sin(np.radians(ls)))
    lat, h = np.radians(lat_deg), np.radians((ltst_hr - 12.0) * 15.0)
    sin_alt = np.sin(lat)*np.sin(dec) + np.cos(lat)*np.cos(dec)*np.cos(h)
    cos_alt = np.sqrt(np.clip(1 - sin_alt**2, 0, 1))
    cos_alt_safe = np.where(cos_alt < 1e-6, 1e-6, cos_alt)
    east  = -np.cos(dec) * np.sin(h) / cos_alt_safe          # cos(alt)*sin(az) / cos(alt)
    north = (np.sin(dec) - sin_alt*np.sin(lat)) / (cos_alt_safe*np.cos(lat))
    return east*cos_alt, north*cos_alt, sin_alt

def solar_const(ls):
    nu = np.radians(ls - 251.0)                   # perihelion near Ls 251
    r = 1.52368 * (1 - 0.0934**2) / (1 + 0.0934*np.cos(nu))
    return 589.0 * (1.52368 / r)**2               # W/m2 at Mars distance

LS = float(ls_deg(DEMO_SOL))
print(f"Ls at landing (sol 0): {float(ls_deg(0)):.1f} deg (expect ~151); Ls at sol {DEMO_SOL}: {LS:.1f} deg; S0 = {solar_const(LS):.0f} W/m2")

Ls at landing (sol 0): 150.7 deg (expect ~151); Ls at sol 800: 228.2 deg; S0 = 707 W/m2


In [11]:
meta = json.loads(META_JSON.read_text())
T_SHAPE = tuple(meta["template_shape"])
T_A, T_B, T_C, T_D, T_E, T_F = meta["template_transform"]      # lon = T_C + col*T_A ; lat = T_F + row*T_E (north-up, so T_B = T_D = 0)
assert T_B == 0 and T_D == 0 and T_E < 0, "template is not a north-up lon/lat grid"
print("template", T_SHAPE, "pixel (deg):", T_A, T_E)

feat = pd.read_csv(FEATURES_CSV)
assert feat.pixel_id.is_unique, "duplicate pixel_id in the features CSV"
assert (feat.pixel_id == feat.row * T_SHAPE[1] + feat.col).all(), "pixel_id does not match row/col for this template shape"
assert feat.row.between(0, T_SHAPE[0]-1).all() and feat.col.between(0, T_SHAPE[1]-1).all(), "row/col outside the template"
print(f"features CSV: {len(feat):,} of {T_SHAPE[0]*T_SHAPE[1]:,} pixels ({len(feat)/(T_SHAPE[0]*T_SHAPE[1]):.1%}); missing pixels become NaN")

def grid(column):
    out = np.full(T_SHAPE, np.nan, "float32")
    out[feat.row.to_numpy(), feat.col.to_numpy()] = feat[column].to_numpy("float32")
    return out

dem, slope, aspect = grid("elev_m"), grid("slope_deg"), grid("aspect_deg")
alb = grid("albedo_mean")          # mean of MY24-26, same as the original demo
ti = grid("ti_themis_tiu")         # THEMIS 100 m only; no TES fallback

rows = np.arange(T_SHAPE[0]); cols = np.arange(T_SHAPE[1])
lat = (T_F + (rows + 0.5) * T_E)[:, None] * np.ones((1, T_SHAPE[1]), "float32")   # from the transform, not the CSV's rounded lon/lat
lon = (T_C + (cols + 0.5) * T_A)[None, :] * np.ones((T_SHAPE[0], 1), "float32")
for n, a in dict(dem=dem, slope=slope, aspect=aspect, albedo=alb, ti=ti).items():
    print(f"{n:7s} min {np.nanmin(a):9.2f}  max {np.nanmax(a):9.2f}  NaN {100*np.mean(~np.isfinite(a)):.1f}%")

template (1436, 1477) pixel (deg): 0.001687051876845213 -0.0016870993139129456
features CSV: 2,120,824 of 2,120,972 pixels (100.0%); missing pixels become NaN
dem     min  -4715.40  max   1115.87  NaN 0.0%
slope   min      0.00  max     82.10  NaN 0.0%
aspect  min      0.00  max    360.00  NaN 0.0%
albedo  min      0.10  max      0.27  NaN 0.0%
ti      min    102.23  max    818.64  NaN 0.0%


In [12]:
# ---- Tau for this sol ----
tau = TAU
if TAU_CSV is not None:
    t = pd.read_csv(TAU_CSV).set_index("sol").tau.sort_index()
    tau = float(np.interp(DEMO_SOL, t.index, t.values))
    print("tau interpolated from CSV:", tau)
else:
    print("TAU is the placeholder", TAU, "- replace with the real value for this sol.")

# ---- Tilted-surface insolation for each 30-min bin ----
S0 = solar_const(LS)
sl, asr = np.radians(slope), np.radians(aspect)
nE, nN, nU = np.sin(sl)*np.sin(asr), np.sin(sl)*np.cos(asr), np.cos(sl)    # aspect = direction the slope faces, deg clockwise from N

ltst_centers = day.ltst_bin_center_hr.values
flux = np.zeros((len(ltst_centers),) + T_SHAPE, "float32")
for k, hr in enumerate(ltst_centers):
    sE, sN, sU = sun_vector(LS, lat, hr)
    cos_i = np.clip(nE*sE + nN*sN + nU*sU, 0, None)
    up = sU > 0.02
    direct = np.where(up, S0 * cos_i * np.exp(-tau / np.clip(sU, 0.05, 1)), 0)
    diffuse = np.where(up, 0.15 * S0 * sU * (1 - np.exp(-tau)) * (1 + nU) / 2, 0)   # crude sky term
    flux[k] = direct + diffuse
print("peak flux over domain (W/m2):", float(np.nanmax(flux)))

TAU is the placeholder 0.5 - replace with the real value for this sol.
peak flux over domain (W/m2): 464.4277038574219


In [13]:
# ---- Heuristic likelihood. ALL weights are assumptions, not fitted. ----
# gap proxy ~ absorbed flux / sqrt(thermal inertia): strong surface heating over a low-inertia surface => big ground-air gap => convection
gap_proxy = (1 - alb)[None] * flux                                   # absorbed flux
gap_proxy /= np.sqrt(np.clip(ti, 50, None))[None]                    # in place: no extra full-size copies

# calibrate the proxy to the rover's observed ground-minus-air gap on this sol
c = (day.longitude_deg_e[0] - T_C) / T_A; r = (day.latitude_deg[0] - T_F) / T_E; r, c = int(r), int(c)   # col from lon, row from lat
assert 0 <= r < T_SHAPE[0] and 0 <= c < T_SHAPE[1], "rover pixel is outside the template"
gap_rover_proxy = gap_proxy[:, r, c]
obs = day.gap_obs_k.values
ok = np.isfinite(obs) & (gap_rover_proxy > 0)
assert ok.any(), "no usable bins at the rover pixel"
scale = float(np.nanmax(obs[ok]) / np.nanmax(gap_rover_proxy[ok]))
print(f"rover pixel (row {r}, col {c}), pixel_id {r*T_SHAPE[1]+c}; meta.json says row {meta['rover_row']}, col {meta['rover_col']}")
print(f"DEM at rover pixel {dem[r, c]:.1f} m vs REMS elevation {day.elevation_m[0]:.1f} m")
print(f"proxy -> kelvin scale {scale:.3f}")
x, y = gap_rover_proxy[ok].astype("float64"), obs[ok]
print("correlation of modeled vs observed gap through the day at the rover pixel: %.2f" % (np.mean((x - x.mean()) * (y - y.mean())) / (x.std() * y.std())))   # no np.corrcoef: it calls BLAS

gap_proxy *= np.float32(scale)                                       # now kelvin; float32 on purpose (a float64 scale would double the memory)
gap_k = gap_proxy
gap_rover_model = gap_k[:, r, c].copy()
flat = np.exp(-(np.nan_to_num(slope, nan=90) / SLOPE_SCALE_DEG)**2).astype("float32")   # steep slopes suppress vortices
gap_ref = np.float32(np.nanpercentile(gap_k, 99.5))
score_t = gap_k / gap_ref
np.clip(score_t, 0, 1, out=score_t)
score_t *= flat[None]                                                # per bin, 0-1
del gap_k, gap_proxy

dm = (ltst_centers >= DAY_LTST[0]) & (ltst_centers <= DAY_LTST[1])
likelihood_mean = score_t[dm].mean(axis=0)
likelihood_max = score_t.max(axis=0)
likelihood_mean[~np.isfinite(ti)] = np.nan
print("score range (daily mean):", np.nanmin(likelihood_mean), np.nanmax(likelihood_mean))

rover pixel (row 304, col 372), pixel_id 449380; meta.json says row 304, col 372
DEM at rover pixel -4437.0 m vs REMS elevation -4461.2 m
proxy -> kelvin scale 1.756
correlation of modeled vs observed gap through the day at the rover pixel: 0.97
score range (daily mean): 1.3826292e-30 0.91571933


In [14]:
np.savez_compressed(NPZ, likelihood_mean=likelihood_mean.astype("float32"), likelihood_max=likelihood_max.astype("float32"),
                    sol=DEMO_SOL, ls_deg=LS, tau=tau, day_ltst=np.array(DAY_LTST), rover_row=r, rover_col=c)
print("wrote", NPZ)

# per-bin table at the rover pixel (join-ready: sol, bin, LTST, Ls, tau, observed + modeled)
tbl = day[["sol","bin","ltst_bin_center_hr","air_temp_k","ground_temp_k","pressure_pa","gap_obs_k"]].copy()
tbl["ls_deg"], tbl["tau"] = LS, tau
tbl["gap_model_k_rover"] = gap_rover_model; tbl["score_rover"] = score_t[:, r, c]
tbl.to_csv(PROC / f"rover_sol{DEMO_SOL}_table.csv", index=False)
tbl.head()

wrote processed_data\gale_dust_devil_likelihood_sol800.npz


,sol,bin,ltst_bin_center_hr,air_temp_k,ground_temp_k,pressure_pa,gap_obs_k,ls_deg,tau,gap_model_k_rover,score_rover
0,800,1,0.38,216.72,209.20,914.57,-7.52,228.182671,0.5,0.0,0.0
1,800,2,0.62,216.72,209.93,915.08,-6.79,228.182671,0.5,0.0,0.0
2,800,5,1.38,216.43,207.11,915.60,-9.32,228.182671,0.5,0.0,0.0
3,800,6,1.62,216.28,208.20,915.41,-8.08,228.182671,0.5,0.0,0.0
4,800,9,2.38,214.42,205.55,913.27,-8.87,228.182671,0.5,0.0,0.0
